# exp054_pseudo_tail_seed_bagging_inference_submit inference

Fit three exp051-style LightGBM capacity pseudo-tail models with different seeds, average raw predictions, and generate a submission with exp025-selected fixed bucket shrink.


## Contents

1. Setup and configuration
2. Selected inference candidate
3. Fit pseudo-tail model and generate submission
4. Artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from pseudo_tail_augmentation import (
    generate_pseudo_tail_submission,
    load_yaml,
    selected_training_variant,
    test_files,
    train_files,
)
from settings import EXPERIMENT_NAME, ExperimentPaths

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_yaml(Path("config.yaml"))

print("Experiment:", EXPERIMENT_NAME)
print("Root:", paths.root)
print("Train data:", paths.train_data_dir)
print("Test data:", paths.test_data_dir)
print("Sample submission:", paths.sample_submission_path)
print("Submission path:", paths.submission_path)
print("Artifacts:", paths.artifacts_dir)
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)


## 2. Selected inference candidate


In [ ]:
variant = selected_training_variant(config)
model_files = train_files(paths, MAX_WELLS if DEBUG else MAX_WELLS)
infer_files = test_files(paths)
print("Selected training variant:", variant["name"])
print("Cutoffs per well:", variant.get("cutoffs_per_well"))
print("Distance balanced:", variant.get("distance_balanced"))
print("Selected postprocess:", config["postprocess"].get("selected_method"))
print("Train wells:", len(model_files))
print("Test wells:", len(infer_files))
print("Final row cap:", config["model"]["training"].get("max_train_rows_final"))
print("Rows per well cap:", config["model"]["training"].get("max_train_rows_per_well"))


## 3. Fit pseudo-tail model and generate submission


In [ ]:
summary = generate_pseudo_tail_submission(
    paths,
    config,
    max_wells=MAX_WELLS if DEBUG else MAX_WELLS,
)
print(json.dumps(summary, indent=2, sort_keys=True))


## 4. Artifacts


In [ ]:
for path in sorted(paths.artifacts_dir.glob("*")):
    if path.is_file():
        print(path.name, path.stat().st_size)
print("Submission exists:", paths.submission_path.exists())
if paths.submission_path.exists():
    print(paths.submission_path)
    print(paths.submission_path.stat().st_size)
